In [1]:
import numpy as np
from scipy.stats import norm, t
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

<div style="background-color:#dbeafe; padding:15px; border-radius:8px; border-left:5px solid #2563eb;">

### **Q1. Two-Proportion Z-Test**

**Control:** 180 conversions / 2,500 users &nbsp;|&nbsp; **Treatment:** 210 conversions / 2,500 users

Perform a full two-proportion z-test. Compute the pooled proportion, standard error, z-statistic, and p-value. State your conclusion at **α = 0.05**.

</div>

<div style="background-color:#f8fafc; padding:13px; border-radius:6px; border-left:4px solid #2563eb; margin-bottom:8px;">

**Hypotheses:**  
H0: p1 = p2 (no difference in conversion rates)  
H1: p2 > p1 (treatment converts better) -- one-tailed test

| Step | Formula |
|------|---------|
| Pooled proportion | $\hat{p} = \dfrac{x_1 + x_2}{n_1 + n_2}$ |
| Standard error | $SE = \sqrt{\hat{p}(1-\hat{p})\left(\dfrac{1}{n_1}+\dfrac{1}{n_2}\right)}$ |
| Z-statistic | $z = \dfrac{\hat{p}_2 - \hat{p}_1}{SE}$ |
| p-value (one-tailed) | $p = P(Z > z)$ |

Under H0 both groups share a common proportion. We pool both samples to estimate it, then measure how many standard errors the observed difference is away from zero.

</div>

In [2]:
# Given data
x1 = 180   # conversions in control
n1 = 2500  # users in control
x2 = 210   # conversions in treatment
n2 = 2500  # users in treatment
alpha = 0.05

In [3]:
# Individual conversion rates
p1 = x1/n1
p2 = x2/n2

In [4]:
# Pooled proportion: best estimate of p under H0
p_pool = (x1+x2)/(n1+n2)
p_pool

0.078

In [5]:
# Standard error of the difference in proportions
se = np.sqrt(p_pool * (1-p_pool) * (1/n1 + 1/n2))
se

np.float64(0.007585037903662711)

In [6]:
# Z-statistic
z_test = (p2-p1)/se
z_test

np.float64(1.5820619688934416)

In [7]:
# Critical value for one-tailed test at alpha = 0.05
z_crit = norm.ppf(1-alpha)
z_crit

np.float64(1.6448536269514722)

In [8]:
# p-value
pval = norm.sf(z_test)
pval

np.float64(0.05681771225610912)

<div style="background-color:#f0fdf4; padding:12px; border-radius:6px; border-left:4px solid #16a34a;">

**Conclusion**

z_test (1.582) &lt; z_crit (1.645) &nbsp;&nbsp;|&nbsp;&nbsp; p-value (0.057) &gt; alpha (0.05)

**Fail to reject H0.** The 30-conversion lift is not statistically significant at the 5% level -- the treatment shows a trend but the evidence is insufficient to conclude a real difference.

</div>

<div style="background-color:#dbeafe; padding:15px; border-radius:8px; border-left:5px solid #2563eb;">

### **Q2. Sample Size Calculation**

Your e-commerce site has a baseline conversion rate of 8%. You want to detect a lift of 1.5 percentage points with 80% power and α = 0.05 (two-tailed). Calculate the required sample size per group. How does your answer change if you want 90% power instead?

</div>

<div style="background-color:#f8fafc; padding:13px; border-radius:6px; border-left:4px solid #2563eb; margin-bottom:8px;">

**Hypotheses:**  
H0: p1 = p2 (no difference)  
H1: p1 != p2 (two-tailed; we want to detect any 1.5 pp shift)

| Component | Formula |
|-----------|--------|
| Average proportion | $\bar{p} = (p_1 + p_2)/2$ |
| Numerator | $\left(z_{\alpha/2}\sqrt{2\bar{p}(1-\bar{p})} + z_{\beta}\sqrt{p_1(1-p_1)+p_2(1-p_2)}\right)^2$ |
| Sample size per group | $n = \text{numerator} / (p_2 - p_1)^2$ |

Higher power (80% to 90%) reduces beta, increasing z_beta and therefore requiring more users per group.

</div>

In [9]:
p1 = 0.08
p2 = 0.095
p = (p1+p2)/2
alpha = 0.05

<div style="background-color:#f1f5f9; padding:8px 13px; border-radius:5px; border-left:3px solid #64748b; margin:4px 0;">

**Case 1 - 80% Power**

</div>

In [10]:
z_crit = norm.ppf(1-alpha/2)  # two-tailed critical value at alpha=0.05
z_crit

np.float64(1.959963984540054)

In [11]:
beta = 0.20               # beta = 1-power 
z_beta = norm.ppf(1-beta) # z-score for 80% power
z_beta

np.float64(0.8416212335729143)

In [12]:
num = ((z_crit * np.sqrt(2*p*(1-p))) + (z_beta * ( np.sqrt( p1 * (1-p1) + p2 * (1 - p2)))))**2  # numerator of sample size formula
deno = (p2-p1)**2  # denominator: squared effect size

In [13]:
n = num/deno  # required sample size per group at 80% power
n

np.float64(5569.345284898933)

<div style="background-color:#f1f5f9; padding:8px 13px; border-radius:5px; border-left:3px solid #64748b; margin:4px 0;">

**Case 2 - 90% Power**

</div>

In [14]:
beta = 0.10
z_beta = norm.ppf(1-beta)  # z-score for 90% power; larger than 80% case
z_beta

np.float64(1.2815515655446004)

In [15]:
num = ((z_crit * np.sqrt(2*p*(1-p))) + (z_beta * ( np.sqrt( p1 * (1-p1) + p2 * (1 - p2)))))**2  # numerator recalculated with 90% power z-score
deno = (p2-p1)**2  # denominator unchanged: same effect size

In [16]:
n2 = num/deno  # required sample size per group at 90% power -- ~34% more users than 80% case
n2

np.float64(7455.2743390574915)

<div style="background-color:#f0fdf4; padding:12px; border-radius:6px; border-left:4px solid #16a34a;">

**Conclusion**

80% power: **n = 5,570 per group** &nbsp;&nbsp;|&nbsp;&nbsp; 90% power: **n = 7,456 per group**

Increasing power from 80% to 90% requires approximately 34% more users per group, reflecting the higher z_beta needed to reduce the false-negative rate.

</div>

<div style="background-color:#dbeafe; padding:15px; border-radius:8px; border-left:5px solid #2563eb;">

### **Q3. Confidence Interval for Difference in Proportions**

You run an A/B test on email subject lines. Open rates: Control = 22% (n = 1,800), Treatment = 25% (n = 1,800). Compute the 95% confidence interval for the difference in proportions. Does it include zero? What do you conclude?

</div>

<div style="background-color:#f8fafc; padding:13px; border-radius:6px; border-left:4px solid #2563eb; margin-bottom:8px;">

**Hypotheses:**  
H0: p2 - p1 = 0 (no difference in open rates)  
H1: p2 - p1 != 0 (two-tailed; treatment open rate is different)

| Step | Formula |
|------|---------|
| Standard error (CI) | $SE = \sqrt{\dfrac{p_1(1-p_1)}{n_1} + \dfrac{p_2(1-p_2)}{n_2}}$ |
| Margin of error | $MOE = z_{\alpha/2} \times SE$ |
| 95% CI | $(p_2 - p_1) \pm MOE$ |

For a CI we do not assume H0 is true, so we use each group's own proportion separately instead of a pooled estimate.

</div>

In [17]:
p1 = 0.22   # control open rate
n1 = 1800   # control group size
p2 = 0.25   # treatment open rate
n2 = 1800   # treatment group size
alpha = 0.05

In [18]:
diff = p2 - p1  # observed difference in open rates

In [19]:
se = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)  # SE for CI uses individual proportions, not pooled
se

np.float64(0.014124446891825536)

In [20]:
z_crit = norm.ppf(1-alpha/2)  # two-tailed critical value at 95% confidence
z_crit

np.float64(1.959963984540054)

In [21]:
moe = z_crit * se  # margin of error

In [22]:
lo = diff - moe  # lower bound of 95% CI
hi = diff + moe  # upper bound of 95% CI
lo, hi

(np.float64(0.002316592790473238), np.float64(0.05768340720952676))

<div style="background-color:#f0fdf4; padding:12px; border-radius:6px; border-left:4px solid #16a34a;">

**Conclusion**

95% CI for (p2 - p1): **(0.0029, 0.0571)** -- does not include zero

**Reject H0.** The treatment subject line has a statistically significant higher open rate; the true difference is estimated between 0.3 and 5.7 percentage points.

</div>

<div style="background-color:#dbeafe; padding:15px; border-radius:8px; border-left:5px solid #2563eb;">

### **Q4. Welch's Two-Sample t-Test**

**Control:** mean = ₹420, std = ₹95, n = 600 &nbsp;|&nbsp; **Treatment:** mean = ₹445, std = ₹110, n = 600

Perform a two-sample t-test using Welch's formula. Calculate the t-statistic, degrees of freedom, and conclude at **α = 0.05**.

</div>

<div style="background-color:#f8fafc; padding:13px; border-radius:6px; border-left:4px solid #2563eb; margin-bottom:8px;">

**Hypotheses:**  
H0: μ1 = μ2 (no difference in mean revenue per user)  
H1: μ1 != μ2 (two-tailed; treatment changes mean revenue)

| Step | Formula |
|------|---------|
| Variance ratios | $r_1 = s_1^2/n_1, \quad r_2 = s_2^2/n_2$ |
| t-statistic | $t = \dfrac{\bar{x}_2 - \bar{x}_1}{\sqrt{r_1 + r_2}}$ |
| Welch df | $df = \dfrac{(r_1+r_2)^2}{\dfrac{r_1^2}{n_1-1}+\dfrac{r_2^2}{n_2-1}}$ |
| p-value (one-tailed) | $p = P(T > t)$  |

Welch's formula does not assume equal variances; it corrects df downward when group variances differ, making the test conservative.

</div>

In [23]:
x1, s1, n1 = 420, 95, 600   # control: mean revenue, std dev, sample size
x2, s2, n2 = 445, 110, 600  # treatment: mean revenue, std dev, sample size
alpha = 0.05

In [24]:
ratio1 = s1**2/n1  # variance / n for control group
ratio2 = s2**2/n2  # variance / n for treatment group

In [25]:
t_stat = (x2-x1)/np.sqrt(ratio1+ratio2)  # t-statistic: difference in means divided by pooled SE
t_stat

np.float64(4.213250442347432)

In [26]:
num = (ratio1+ratio2)**2                      # numerator of Welch df formula
den = (ratio1**2/(n1-1) + ratio2**2/(n2-1))  # denominator accounts for each group's variance
df = num/den                                   # Welch-Satterthwaite degrees of freedom
df

1173.1430534564715

In [27]:
t_crit = t.ppf(1-alpha, df)  # one-tailed critical value from t-distribution at computed df
t_crit

np.float64(1.646153537012905)

In [28]:
pval = t.sf(t_stat, df)  # p-value: P(T > t_stat) under H0 using survival function
pval

np.float64(1.355068840065489e-05)

<div style="background-color:#f0fdf4; padding:12px; border-radius:6px; border-left:4px solid #16a34a;">

**Conclusion**

t_stat (4.213) > t_crit (1.646) &nbsp;&nbsp;|&nbsp;&nbsp; p-value (0.0000136) < alpha (0.05)

**Reject H0.** The ₹25 difference in mean revenue per user is statistically significant; the treatment group generates meaningfully higher revenue.

</div>